# ⚙️ Construcción del Pipeline de Transformación
## Dataset: Messy Clinic Appointments

**Objetivo:** Ensamblar todos los transformers del módulo `src/` en un Pipeline
de Scikit-Learn reproducible, ejecutarlo sobre el CSV crudo y verificar que la
salida es una matriz numérica limpia, lista para un modelo de ML.

**¿Por qué un Pipeline de Scikit-Learn?**
- Garantiza que el mismo proceso se aplique en entrenamiento y producción.
- Evita Data Leakage: los parámetros (medianas, límites IQR) se aprenden solo
  con datos de entrenamiento y se aplican en los de prueba.
- Permite visualizar la arquitectura completa con un solo comando.


## 0. Configuración del entorno

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import os
sys.path.append('..')  # Para importar desde src/

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn import set_config
set_config(display='diagram')   # Activa el diagrama visual del pipeline

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

from src.config import (
    RAW_CSV, CLEAN_CSV, OUTPUTS_DIR,
    TARGET_COL, NUMERIC_COLS, CATEGORICAL_COLS, COLS_TO_DROP
)
os.makedirs(OUTPUTS_DIR, exist_ok=True)
os.makedirs(CLEAN_CSV.parent, exist_ok=True)

print("Entorno listo")
print(f"   Columnas numéricas:    {NUMERIC_COLS}")
print(f"   Columnas categóricas:  {CATEGORICAL_COLS}")
print(f"   Columnas a eliminar:   {COLS_TO_DROP}")


## 1. Carga del dataset crudo

In [ ]:
df_raw = pd.read_csv(RAW_CSV)
print(f"Dataset crudo: {df_raw.shape[0]} filas × {df_raw.shape[1]} columnas")
df_raw.head(4)


## 2. Separación de Features y Variable Objetivo

Antes de construir el pipeline separamos `X` (features) de `y` (target).
El pipeline solo transforma `X`; `y` se normaliza por separado porque no
debe pasar por imputación ni escalado.


In [ ]:
def normalize_target(series: pd.Series) -> pd.Series:
    """
    Normaliza follow_up_required a binario 0/1.
    Variantes positivas: 'Yes', 'Y', '1'  → 1
    Variantes negativas: 'No',  'N', '0'  → 0
    """
    def _map(x):
        x = str(x).strip().lower()
        if x in ['yes', 'y', '1']: return 1
        if x in ['no',  'n', '0']: return 0
        return np.nan
    return series.apply(_map)

y = normalize_target(df_raw[TARGET_COL])
X = df_raw.drop(columns=[TARGET_COL])

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"Nulos en y tras normalización: {y.isna().sum()}")
print()
print("Distribución de y:")
dist = y.value_counts()
for val, cnt in dist.items():
    label = 'Requiere seguimiento' if val == 1 else 'No requiere'
    print(f"  {val} ({label}): {cnt} ({cnt/len(y)*100:.1f}%)")


## 3. Importar los Transformers personalizados

In [ ]:
from src.transformers import (
    # Transformers genéricos (reutilizados)
    DropColumnsTransformer,
    DropHighMissingTransformer,
    OutlierCapper,
    DropZeroVarianceTransformer,
    SmartImputerTransformer,
    # Transformers específicos del dataset clínico (nuevos)
    GenderNormalizerTransformer,
    BillingCleanerTransformer,
    DateFeatureTransformer,
)

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

print("Todos los transformers importados correctamente")


## 4. Construcción del Pipeline

El pipeline se divide en dos niveles:

**Nivel 1 — Pre-limpieza global** (transforma todo el DataFrame):
aplica los transformers que necesitan ver todas las columnas a la vez.

**Nivel 2 — ColumnTransformer** (bifurca en dos rutas paralelas):
- Ruta numérica: imputa → capping → escalar
- Ruta categórica: imputa → codificar

Esta separación es necesaria porque los números y las categorías requieren
tratamientos fundamentalmente distintos.


In [ ]:
from src.config import HIGH_MISSING_THRESHOLD, SIMPLE_IMPUTE_THRESHOLD

# ── Ruta NUMÉRICA ─────────────────────────────────────────────────────
# Variables: age, billing_amount, waiting_days, appointment_dow
numeric_pipeline = Pipeline([
    # 1. Imputar primero: billing_amount tiene 50 nulos (5%)
    #    El OutlierCapper no puede operar con NaN, debe ir después
    ('imputer',       SmartImputerTransformer(low_threshold=SIMPLE_IMPUTE_THRESHOLD)),

    # 2. Limitar outliers con IQR (Winsorización)
    ('capper',        OutlierCapper(apply_capping=True)),

    # 3. Eliminar columnas constantes (varianza = 0)
    ('zero_variance', DropZeroVarianceTransformer()),

    # 4. Escalar a media=0, desviación estándar=1
    ('scaler',        StandardScaler()),
])

# ── Ruta CATEGÓRICA ───────────────────────────────────────────────────
# Variables: gender, department
categorical_pipeline = Pipeline([
    # 1. Imputar nulos con moda (valor más frecuente)
    ('imputer', SmartImputerTransformer(low_threshold=SIMPLE_IMPUTE_THRESHOLD)),

    # 2. Convertir categorías a columnas 0/1 (One-Hot Encoding)
    #    handle_unknown='ignore': si en producción llega un valor nuevo,
    #    no lanza error sino que lo trata como todos ceros
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])

# ── ColumnTransformer: aplica ambas rutas en paralelo ─────────────────
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_pipeline,    NUMERIC_COLS),
        ('cat', categorical_pipeline, CATEGORICAL_COLS),
    ],
    remainder='drop'  # descarta cualquier columna no listada
)

# ── Pipeline principal: pre-limpieza + ColumnTransformer ──────────────
pipeline_completo = Pipeline([
    # Paso 1: normalizar género (8 variantes → Male/Female)
    ('gender_norm',    GenderNormalizerTransformer()),

    # Paso 2: extraer número de billing_amount ('£425.8' → 425.8)
    ('billing_clean',  BillingCleanerTransformer()),

    # Paso 3: parsear fechas → generar waiting_days y appointment_dow
    ('date_features',  DateFeatureTransformer()),

    # Paso 4: eliminar columnas sin valor predictivo
    ('drop_cols',      DropColumnsTransformer(COLS_TO_DROP)),

    # Paso 5: descartar columnas con >80% nulos (si las hubiera)
    ('drop_missing',   DropHighMissingTransformer(threshold=HIGH_MISSING_THRESHOLD)),

    # Paso 6: rutas paralelas para numéricas y categóricas
    ('preprocessor',   preprocessor),
])

print("Pipeline construido")


## 5. Diagrama Visual del Pipeline

Scikit-Learn genera automáticamente un diagrama HTML interactivo del pipeline.
Cada paso es una caja que puedes inspectar.


In [ ]:
# Esto renderiza el diagrama directamente en Jupyter
pipeline_completo


## 6. Ejecución: fit_transform sobre el dataset completo

`fit_transform(X)` hace dos cosas en una sola llamada:
1. **fit**: aprende los parámetros de cada paso (medianas, límites IQR, etc.)
2. **transform**: aplica las transformaciones

Esto es equivalente a llamar `.fit(X).transform(X)`, pero más eficiente.


In [ ]:
print("Dimensiones originales:", X.shape)
print()

X_procesado = pipeline_completo.fit_transform(X)

print()
print("=" * 50)
print(f"Dimensiones procesadas: {X_procesado.shape}")
print(f"Tipo de salida: {type(X_procesado)}")
print(f"Valores NaN en la salida: {np.isnan(X_procesado).sum()}")
print(f"Tipo de datos: {X_procesado.dtype}")


## 7. Features Resultantes

Después del pipeline, las columnas originales se transformaron en una nueva
matriz. Aquí vemos exactamente qué feature corresponde a cada columna.


In [ ]:
nombres = pipeline_completo.named_steps['preprocessor'].get_feature_names_out()

print(f"Total de features para el modelo: {len(nombres)}\n")

# Mostrar con separación por tipo
print("── Numéricas (escaladas) ──────────────────")
for n in nombres:
    if n.startswith('num__'):
        print(f"  {n.replace('num__', '')}")

print()
print("── Categóricas (OneHot) ───────────────────")
for n in nombres:
    if n.startswith('cat__'):
        print(f"  {n.replace('cat__', '')}")


## 8. Vista del Dataset Procesado

In [ ]:
nombres_limpios = [n.replace('num__','').replace('cat__','') for n in nombres]
df_procesado = pd.DataFrame(X_procesado, columns=nombres_limpios)

print("Primeras 5 filas de la matriz lista para el modelo:")
df_procesado.head()


In [ ]:
print("Estadísticas del dataset procesado:")
df_procesado.describe().round(3)


In [ ]:
# Verificación final de calidad
print("=== Verificación de calidad del dataset procesado ===\n")

checks = {
    "Sin valores nulos":       df_procesado.isnull().sum().sum() == 0,
    "Sin columnas constantes": df_procesado.std().min() > 0,
    "Variables numéricas escaladas (media ≈ 0)":
        abs(df_procesado[['age','billing_amount','waiting_days','appointment_dow']].mean()).max() < 0.01,
    "Columnas binarias en rango [0,1]":
        df_procesado[[c for c in nombres_limpios if 'gender' in c or 'department' in c]].isin([0,1]).all().all(),
}

for descripcion, resultado in checks.items():
    estado = "ok" if resultado else "x"
    print(f"  {estado} {descripcion}")


## 9. Guardar Dataset Procesado

In [ ]:
# Agregamos la variable objetivo al dataset final
df_final = df_procesado.copy()
df_final['follow_up_required'] = y.values

df_final.to_csv(CLEAN_CSV, index=False)
print(f"Dataset procesado guardado en: {CLEAN_CSV}")
print(f"   Shape final: {df_final.shape}")
print(f"   Columnas: {df_final.columns.tolist()}")


## 10. Comparación: Antes vs. Después

Resumen visual del impacto del pipeline.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Antes: distribución de billing_amount con símbolos
import re
billing_original = (
    df_raw['billing_amount']
    .astype(str)
    .str.extract(r'([\\d.]+)')[0]
    .astype(float)
    .dropna()
)
axes[0].hist(billing_original, bins=30, color='#E07B54', edgecolor='white', alpha=0.8)
axes[0].set_title('billing_amount — ANTES\n(texto con símbolos de moneda, 50 nulos)', fontsize=11)
axes[0].set_xlabel('Valor extraído')
axes[0].set_ylabel('Frecuencia')

# Después: billing_amount escalado
axes[1].hist(df_procesado['billing_amount'], bins=30, color='#5B8DB8',
             edgecolor='white', alpha=0.8)
axes[1].set_title('billing_amount — DESPUÉS\n(escalado: media=0, std=1, sin nulos)', fontsize=11)
axes[1].set_xlabel('Valor escalado (StandardScaler)')
axes[1].axvline(0, color='#E07B54', linestyle='--', linewidth=1.5, label='Media = 0')
axes[1].legend()

plt.suptitle('Efecto del Pipeline sobre billing_amount', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(OUTPUTS_DIR / 'pipeline_before_after.png', dpi=150, bbox_inches='tight')
plt.show()


## 10. Validación Visual: Antes vs. Después del Pipeline

Una sección de validación cuantifica el impacto real de cada transformación.
No basta con decir "limpiamos los datos" — hay que mostrarlo con números y gráficos.


In [ ]:
# Calcular stat previa necesaria para el reporte
billing_raw_num2 = (df_raw['billing_amount'].astype(str)
                    .str.extract(r'([0-9.]+)')[0].astype(float))
X_procesado_stats_before = billing_raw_num2.mean()
print(f"Media billing antes (valor numérico puro, sin conversión): ${X_procesado_stats_before:.2f}")


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# ── Fila 1: Nulos ─────────────────────────────────────────────────────
nulos_antes  = df_raw[['gender','billing_amount']].isnull().sum()
nulos_despues = pd.Series({'gender': 0, 'billing_amount': 0})

x = np.arange(2)
w = 0.35
axes[0,0].bar(x - w/2, nulos_antes.values,  w, label='Antes',  color='#E07B54', edgecolor='white')
axes[0,0].bar(x + w/2, nulos_despues.values, w, label='Después', color='#5B8DB8', edgecolor='white')
axes[0,0].set_xticks(x)
axes[0,0].set_xticklabels(['gender', 'billing_amount'])
axes[0,0].set_title('Valores Nulos: Antes vs. Después', fontsize=11)
axes[0,0].set_ylabel('Cantidad de nulos')
axes[0,0].legend()
for i, v in enumerate(nulos_antes.values):
    axes[0,0].text(i - w/2, v + 0.5, str(v), ha='center', fontsize=10)
    axes[0,0].text(i + w/2, 0.5, '0', ha='center', fontsize=10)

# ── Fila 1: Variantes de género ───────────────────────────────────────
gender_antes  = df_raw['gender'].value_counts(dropna=False)
GENDER_MAP = {'male':'Male','m':'Male','1':'Male','female':'Female','f':'Female','0':'Female'}
gender_despues = df_raw['gender'].str.strip().str.lower().map(GENDER_MAP).value_counts(dropna=False)

labels_a = [str(v) if str(v) != 'nan' else 'NaN' for v in gender_antes.index]
axes[0,1].bar(range(len(gender_antes)), gender_antes.values, color='#E07B54', edgecolor='white')
axes[0,1].set_xticks(range(len(gender_antes)))
axes[0,1].set_xticklabels(labels_a, rotation=45, ha='right', fontsize=9)
axes[0,1].set_title(f'Gender ANTES\n({len(gender_antes)} variantes distintas)', fontsize=11)
axes[0,1].set_ylabel('Registros')

labels_d = [str(v) if str(v) != 'nan' else 'NaN' for v in gender_despues.index]
colors_d = ['#E07B54' if l=='Female' else '#5B8DB8' if l=='Male' else '#CCCCCC' for l in labels_d]
axes[0,2].bar(range(len(gender_despues)), gender_despues.values, color=colors_d, edgecolor='white')
axes[0,2].set_xticks(range(len(gender_despues)))
axes[0,2].set_xticklabels(labels_d, fontsize=11)
axes[0,2].set_title(f'Gender DESPUÉS\n(2 valores + NaN imputado)', fontsize=11)
axes[0,2].set_ylabel('Registros')
for i, v in enumerate(gender_despues.values):
    axes[0,2].text(i, v + 1, str(v), ha='center', fontsize=10)

# ── Fila 2: billing_amount — distribución antes vs después ────────────
billing_raw_num = (df_raw['billing_amount'].astype(str)
                   .str.extract(r'([0-9.]+)')[0].astype(float).dropna())
axes[1,0].hist(billing_raw_num, bins=30, color='#E07B54', edgecolor='white', alpha=0.8)
axes[1,0].set_title('billing_amount ANTES\n(texto con monedas mixtas, 50 nulos)', fontsize=11)
axes[1,0].set_xlabel('Valor extraído sin conversión')
axes[1,0].set_ylabel('Frecuencia')

axes[1,1].hist(df_procesado['billing_amount'], bins=30, color='#5B8DB8', edgecolor='white', alpha=0.8)
axes[1,1].set_title('billing_amount DESPUÉS\n(USD unificado, escalado, sin nulos)', fontsize=11)
axes[1,1].set_xlabel('Valor escalado (StandardScaler)')
axes[1,1].axvline(0, color='#E07B54', linestyle='--', linewidth=1.5, label='Media=0')
axes[1,1].legend()

# ── Fila 2: tabla resumen ─────────────────────────────────────────────
resumen = {
    'Métrica': ['Filas', 'Columnas', 'Valores nulos', 'Variantes en gender',
                'Variantes en follow_up', 'Tipo de billing_amount'],
    'Antes':   [1000, 10, 100, 8, 6, 'texto (str)'],
    'Después': [1000, 10, 0, 2, 2, 'float (USD)'],
}
df_resumen = pd.DataFrame(resumen)
axes[1,2].axis('off')
tabla = axes[1,2].table(
    cellText=df_resumen.values,
    colLabels=df_resumen.columns,
    cellLoc='center',
    loc='center',
    bbox=[0, 0, 1, 1]
)
tabla.auto_set_font_size(False)
tabla.set_fontsize(10)
for (r, c), cell in tabla.get_celld().items():
    if r == 0:
        cell.set_facecolor('#2C3E50')
        cell.set_text_props(color='white', fontweight='bold')
    elif c == 2 and r > 0:
        cell.set_facecolor('#EAF4FB')
axes[1,2].set_title('Resumen de Impacto del Pipeline', fontsize=11, pad=15)

plt.suptitle('Validación: Impacto Real de la Limpieza', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(OUTPUTS_DIR / 'validation_before_after.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Resumen numérico de la validación
print("=" * 55)
print("       REPORTE DE VALIDACIÓN DEL PIPELINE")
print("=" * 55)
print(f"  Filas procesadas:          {len(df_procesado):,} (0 perdidas)")
print(f"  Features en salida:        {df_procesado.shape[1]}")
print(f"  Valores nulos:             {df_procesado.isnull().sum().sum()}")
print(f"  Media billing (antes):     ${X_procesado_stats_before:.2f} (mixed currencies)")
print(f"  Media billing (después):   {df_procesado['billing_amount'].mean():.4f} (scaled, ~0)")
print(f"  Variantes gender (antes):  8 + NaN")
print(f"  Variantes gender (después):2 + imputado")
print(f"  Columnas OneHot generadas: {len([c for c in nombres_limpios if 'gender' in c or 'dept' in c])}")
print("=" * 55)
print("Todos los checks de calidad superados")


## 11. Resumen del Pipeline

| Paso | Transformer | Variables afectadas | Resultado |
|---|---|---|---|
| 1 | `GenderNormalizerTransformer` | `gender` | 8 variantes → Male/Female |
| 2 | `BillingCleanerTransformer` | `billing_amount` | Texto → float64 |
| 3 | `DateFeatureTransformer` | `appointment_date`, `booking_date` | Genera `waiting_days`, `appointment_dow` |
| 4 | `DropColumnsTransformer` | `patient_name`, `doctor`, fechas | Eliminadas |
| 5 | `DropHighMissingTransformer` | todas | Elimina cols >80% nulos (ninguna en este dataset) |
| 6a (num) | `SmartImputerTransformer` | `billing_amount` | 50 nulos → mediana |
| 6b (num) | `OutlierCapper` | numéricas | Winsorización IQR |
| 6c (num) | `StandardScaler` | numéricas | Media=0, Std=1 |
| 6d (cat) | `SmartImputerTransformer` | `gender` | 50 nulos → moda |
| 6e (cat) | `OneHotEncoder` | `gender`, `department` | 2 cols → 6 columnas binarias |

**Input:** 1000 filas × 9 columnas (mezcladas, sucias, texto)  
**Output:** 1000 filas × 10 features numéricas limpias, sin nulos, escaladas
